In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem.snowball import SnowballStemmer
from tqdm import tqdm


[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
data = pd.read_excel('data.xlsx')
df = data[[not elem for elem in data.comments.isna()]].reset_index()

texts = df.comments.values.tolist()
labels = df.Label.values

In [ ]:
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
def label_to_name(label):
    if label == 0:
        return 'negative'
    elif label == 1:
        return 'neutral'
    elif label == 2:
        return 'positive'
    else:
        raise ValueError(f'Wrong label {label}')

def name_to_label(label):
    if label == 'negative':
        return 0
    elif label == 'neutral':
        return 1
    elif label == 'positive':
        return 2
    else:
        raise ValueError(f'Wrong label {label}')

## LoRA

In [ ]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from datasets import Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "blanchefort/rubert-base-cased-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token if tokenizer.eos_token else tokenizer.unk_token

sentiment_labels = ["negative", "neutral", "positive"]
label2id = {label: idx for idx, label in enumerate(sentiment_labels)}
id2label = {idx: label for idx, label in enumerate(sentiment_labels)}

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro', zero_division=0)
    return {
        'accuracy': accuracy,
        'precision_macro': precision,
        'recall_macro': recall,
        'f1_macro': f1
    }

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=512,
        return_tensors='pt'
    )

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["query", "value", "key", "dense", "output.dense"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    inference_mode=False
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_metrics = []

for fold, (train_idx, val_idx) in enumerate(skf.split(texts, labels)):
    print(f"\nFold {fold + 1}/5")
    print(f"Train size: {len(train_idx)}, Val size: {len(val_idx)}")

    train_texts = [texts[i] for i in train_idx]
    train_labels = [labels[i] for i in train_idx]
    val_texts = [texts[i] for i in val_idx]
    val_labels = [labels[i] for i in val_idx]

    train_dataset = Dataset.from_dict({
        'text': train_texts,
        'label': train_labels
    })
    val_dataset = Dataset.from_dict({
        'text': val_texts,
        'label': val_labels
    })

    def tokenize_and_format(examples):
        tokenized = tokenizer(
            examples['text'],
            truncation=True,
            padding='max_length',
            max_length=512
        )
        tokenized['labels'] = examples['label']
        return tokenized

    train_dataset = train_dataset.map(tokenize_and_format, batched=True, remove_columns=['text', 'label'])
    val_dataset = val_dataset.map(tokenize_and_format, batched=True, remove_columns=['text', 'label'])

    train_dataset.set_format('torch')
    val_dataset.set_format('torch')

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(sentiment_labels),
        id2label=id2label,
        label2id=label2id,
        trust_remote_code=True
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    model.to(device)

    trainer = Trainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        tokenizer=tokenizer
    )

    trainer.train()

    eval_results = trainer.evaluate()
    print('accuracy': eval_results['eval_accuracy'])
    print('recall macro': eval_results['eval_accuracy'])
    print('precision macro': eval_results['eval_accuracy'])
    print('f1-score macro': eval_results['eval_accuracy'])

    cv_metrics.append({
        'fold': fold + 1,
        'accuracy': eval_results['eval_accuracy'],
        'precision_macro': eval_results['eval_precision_macro'],
        'recall_macro': eval_results['eval_recall_macro'],
        'f1_macro': eval_results['eval_f1_macro']
    })

    del model
    del trainer
    torch.cuda.empty_cache()

results_df = pd.DataFrame(cv_metrics)

Train size: 16942, Val size: 4236
trainable params: 2,681,091 || all params: 180,536,838 || trainable%: 1.4851

Fold 1/5
accuracy: 0.90965
recall macro: 0.86573
precision macro: 0.93637
f1-score macro: 0.90048

Fold 2/5
accuracy: 0.91173
recall macro: 0.86895
precision macro: 0.93892
f1-score macro: 0.89917

Fold 3/5
accuracy: 0.91156
recall macro: 0.86625
precision macro: 0.93815
f1-score macro: 0.89944

Fold 4/5
accuracy: 0.91408
recall macro: 0.86650
precision macro: 0.94143
f1-score macro: 0.90034

Fold 5/5
accuracy: 0.91415
recall macro: 0.87041
precision macro: 0.93840
f1-score macro: 0.90500


In [ ]:
mean_metrics = {
    'fold': 'Mean',
    'accuracy': results_df['accuracy'].mean().round(2),
    'precision_macro': results_df['precision_macro'].mean().round(2),
    'recall_macro': results_df['recall_macro'].mean().round(2),
    'f1_macro': results_df['f1_macro'].mean().round(2)
}


results_df = pd.DataFrame([mean_metrics])
results_df

,fold,accuracy,precision_macro,recall_macro,f1_macro
0,Mean,0.91,0.94,0.89,0.91


# DistillBert + LoRA

In [ ]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from datasets import Dataset
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "Eka-Korn/distillbert-qa-tuned-lora_1.01_v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

sentiment_labels = ["negative", "neutral", "positive"]
label2id = {label: idx for idx, label in enumerate(sentiment_labels)}
id2label = {idx: label for idx, label in enumerate(sentiment_labels)}

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro', zero_division=0)
    return {
        'accuracy': accuracy,
        'precision_macro': precision,
        'recall_macro': recall,
        'f1_macro': f1
    }

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_lin", "v_lin", "k_lin", "out_lin"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    inference_mode=False
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_metrics = []

for fold, (train_idx, val_idx) in enumerate(skf.split(texts, labels)):
    print(f"\nFold {fold + 1}/5")
    print(f"Train size: {len(train_idx)}, Val size: {len(val_idx)}")

    train_texts = [texts[i] for i in train_idx]
    train_labels = [labels[i] for i in train_idx]
    val_texts = [texts[i] for i in val_idx]
    val_labels = [labels[i] for i in val_idx]

    train_dataset = Dataset.from_dict({
        'text': train_texts,
        'label': train_labels
    })
    val_dataset = Dataset.from_dict({
        'text': val_texts,
        'label': val_labels
    })

    def tokenize_and_format(examples):
        tokenized = tokenizer(
            examples['text'],
            truncation=True,
            padding='max_length',
            max_length=512
        )
        tokenized['labels'] = examples['label']
        return tokenized

    train_dataset = train_dataset.map(tokenize_and_format, batched=True, remove_columns=['text', 'label'])
    val_dataset = val_dataset.map(tokenize_and_format, batched=True, remove_columns=['text', 'label'])

    train_dataset.set_format('torch')
    val_dataset.set_format('torch')

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(sentiment_labels),
        id2label=id2label,
        label2id=label2id
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    model.to(device)

    trainer = Trainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        tokenizer=tokenizer
    )

    trainer.train()

    eval_results = trainer.evaluate()
    print('accuracy': eval_results['eval_accuracy'])
    print('recall macro': eval_results['eval_accuracy'])
    print('precision macro': eval_results['eval_accuracy'])
    print('f1-score macro': eval_results['eval_accuracy'])

    cv_metrics.append({
        'fold': fold + 1,
        'accuracy': eval_results['eval_accuracy'],
        'precision_macro': eval_results['eval_precision_macro'],
        'recall_macro': eval_results['eval_recall_macro'],
        'f1_macro': eval_results['eval_f1_macro']
    })

    del model
    del trainer
    torch.cuda.empty_cache()

results_df = pd.DataFrame(cv_metrics)


Fold 1/5
Train size: 16942, Val size: 4236
accuracy: 0.81200
recall macro: 0.69450
precision macro: 0.75278
f1-score macro: 0.72048

Fold 2/5
Train size: 16942, Val size: 4236
accuracy: 0.81393
recall macro: 0.68795
precision macro: 0.75102
f1-score macro: 0.72321

Fold 3/5
Train size: 16942, Val size: 4236
accuracy: 0.80645
recall macro: 0.69042
precision macro: 0.75001
f1-score macro: 0.72410

Fold 4/5
Train size: 16942, Val size: 4236
accuracy: 0.81259
recall macro: 0.68919
precision macro: 0.74650
f1-score macro: 0.71667

Fold 5/5
Train size: 16942, Val size: 4236
accuracy: 0.81336
recall macro: 0.69070
precision macro: 0.74721
f1-score macro: 0.72088
The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [ ]:
mean_metrics = {
    'fold': 'Mean',
    'accuracy': results_df['accuracy'].mean().round(2),
    'precision_macro': results_df['precision_macro'].mean().round(2),
    'recall_macro': results_df['recall_macro'].mean().round(2),
    'f1_macro': results_df['f1_macro'].mean().round(2)
}


results_df = pd.DataFrame([mean_metrics])
results_df

,fold,accuracy,precision_macro,recall_macro,f1_macro
0,Mean,0.81,0.75,0.69,0.72
